In [49]:
import numpy as np
import torch
from datasets import load_dataset


In [50]:
# load the first shard
data_files = {"train": ["data-0000.tar.gz"]}
ds = load_dataset("commaai/commavq", data_files=data_files)


In [51]:
SEGMENT_IDX = 1

In [52]:
# ds["train"][0]

In [53]:
poses = np.array(ds["train"][SEGMENT_IDX]["pose.npy"])
tokens = np.array(ds["train"][SEGMENT_IDX]["token.npy"])
tokens = tokens.reshape(tokens.shape[0], -1).astype(np.int64)
tokens = torch.from_numpy(tokens).to(device="cuda")
tokens


tensor([[ 895,  626,  801,  ..., 1019,  548,  558],
        [ 895,  599,  152,  ..., 1019,  968,  378],
        [1019,  203,  576,  ..., 1019,  521,  781],
        ...,
        [ 663,  363,  663,  ..., 1001,   11,  378],
        [ 274,  363,  801,  ..., 1001,  915,  104],
        [ 663,  363,  801,  ...,  597,   11,  156]], device='cuda:0')

## Visualise


In [54]:
import sys

sys.path.append("..")
import numpy as np
import torch
from IPython.display import Video
from tqdm import trange

from utils.video import transpose_and_clip, write_video
from utils.vqvae import CompressorConfig, Decoder

In [55]:
# load model
config = CompressorConfig()
with torch.device("meta"):
    decoder = Decoder(config)
decoder.load_state_dict_from_url(
    "https://huggingface.co/commaai/commavq-gpt2m/resolve/main/decoder_pytorch_model.bin",
    assign=True,
)
decoder = decoder.eval().to(device="cuda")
# decoder

In [56]:
decoded_video = []
with torch.no_grad():
    for i in trange(len(tokens)):
        decoded = decoder(tokens[i][None])
        decoded_video.append(decoded)
decoded_video = torch.cat(decoded_video, dim=0).cpu().numpy()
decoded_video = transpose_and_clip(decoded_video)

100%|██████████| 1200/1200 [00:06<00:00, 193.99it/s]


In [57]:
save_dst = "../examples/data_set_decoded.mp4"
write_video(decoded_video, save_dst, fps=20)

'../examples/data_set_decoded.mp4'